DATA FUSION


In [ ]:
import os
from PIL import Image
import numpy as np

def tile_padded_image_and_labels(image_path, label_path_1d, label_path_visual, output_dir, tile_size=256, stride=128, anchor_image_path=None):
    # Open images with consistent modes
    image = Image.open(image_path).convert('RGB')  # Ensure RGB mode
    image = np.array(image)
    print(f"Image shape: {image.shape}")  # Debug statement

    label_1d = Image.open(label_path_1d).convert('L')  # Ensure grayscale mode
    label_1d = np.array(label_1d)
    print(f"Label 1D shape: {label_1d.shape}")  # Debug statement

    label_visual = Image.open(label_path_visual).convert('RGB')  # Ensure RGB mode
    label_visual = np.array(label_visual)
    print(f"Label visual shape: {label_visual.shape}")  # Debug statement

    # Load anchor image
    if anchor_image_path:
        anchor_image = Image.open(anchor_image_path).convert('RGB')  # Ensure RGB mode
        anchor_image = np.array(anchor_image)
        print(f"Anchor image shape: {anchor_image.shape}")  # Debug statement
    else:
        anchor_image = image  # Use the current image as anchor if none provided

    # Initialize ImagePadder
    padder = ImagePadder(
        image_shape=image.shape,
        tile_size=tile_size,
        stride=stride,
        anchor_image=anchor_image
    )

    # Pad images and labels
    padded_image = padder.pad_image(image)
    padded_label_1d = padder.pad_label(label_1d)
    padded_label_visual = padder.pad_label(label_visual)

    # Ensure output directories exist
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels_1D'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)

    count = 0
    img_width, img_height = padded_image.shape[1], padded_image.shape[0]

    for i in range(0, img_width - tile_size + 1, stride):
        for j in range(0, img_height - tile_size + 1, stride):
            # Crop the image and labels
            tile_image = padded_image[j:j + tile_size, i:i + tile_size]
            tile_label_1d = padded_label_1d[j:j + tile_size, i:i + tile_size]
            tile_label_visual = padded_label_visual[j:j + tile_size, i:i + tile_size]

            # Save tiles
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            tile_image_name = f"{base_name}_{count}.png"
            tile_label_1d_name = f"{base_name}_{count}_label_1D.png"
            tile_label_visual_name = f"{base_name}_{count}_label.png"

            Image.fromarray(tile_image).save(os.path.join(output_dir, 'images', tile_image_name))
            Image.fromarray(tile_label_1d).save(os.path.join(output_dir, 'labels_1D', tile_label_1d_name))
            Image.fromarray(tile_label_visual).save(os.path.join(output_dir, 'labels', tile_label_visual_name))

            count += 1

    print(f"Tiled {count} patches from {image_path}")

def process_subset1_images_padded(input_dir, output_dir, tile_size=256, stride=128, anchor_image_name="img_0814.jpg"):
    images_dir = os.path.join(input_dir, 'images')
    labels_dir_1d = os.path.join(input_dir, 'labels_1D')
    labels_dir_visual = os.path.join(input_dir, 'labels')

    anchor_image_path = os.path.join(images_dir, anchor_image_name)

    image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg') or f.endswith('.png')]

    for image_file in image_files:
        image_path = os.path.join(images_dir, image_file)
        base_name = os.path.splitext(image_file)[0]

        # Corresponding label files
        label_file_1d = base_name + '.png'  # Assuming labels_1D have the same base name
        label_file_visual = base_name + '.png'  # Assuming labels have the same base name

        label_path_1d = os.path.join(labels_dir_1d, label_file_1d)
        label_path_visual = os.path.join(labels_dir_visual, label_file_visual)

        if os.path.exists(label_path_1d) and os.path.exists(label_path_visual):
            tile_padded_image_and_labels(
                image_path,
                label_path_1d,
                label_path_visual,
                output_dir,
                tile_size=tile_size,
                stride=stride,
                anchor_image_path=anchor_image_path
            )
        else:
            print(f"Label files not found for image {image_file}")

# Example usage:
input_dir_subset1_train = 'H:\Derrame_Data\OilDataset\\train'
output_dir_patches_train = 'H:/Derrame_Data\OilDatasetPatches/train'

tile_size = 256
stride = 128  # 50% overlap

process_subset1_images_padded(
    input_dir=input_dir_subset1_train,
    output_dir=output_dir_patches_train,
    tile_size=tile_size,
    stride=stride
)



In [4]:
input_dir_subset1_test = 'H:\Derrame_Data\OilDataset\\test'
output_dir_patches_test = 'H:/Derrame_Data\OilDatasetPatches/test'

tile_size = 256
stride = 128  # 50% overlap

process_subset1_images_padded(
    input_dir=input_dir_subset1_test,
    output_dir=output_dir_patches_test,
    tile_size=tile_size,
    stride=stride
)


Image shape: (650, 1250, 3)
Label 1D shape: (650, 1250)
Label visual shape: (650, 1250, 3)
Anchor image shape: (650, 1250, 3)
Tiled 45 patches from H:\Derrame_Data\OilDataset\test\images\img_0001.jpg
Image shape: (650, 1250, 3)
Label 1D shape: (650, 1250)
Label visual shape: (650, 1250, 3)
Anchor image shape: (650, 1250, 3)
Tiled 45 patches from H:\Derrame_Data\OilDataset\test\images\img_0002.jpg
Image shape: (650, 1250, 3)
Label 1D shape: (650, 1250)
Label visual shape: (650, 1250, 3)
Anchor image shape: (650, 1250, 3)
Tiled 45 patches from H:\Derrame_Data\OilDataset\test\images\img_0003.jpg
Image shape: (650, 1250, 3)
Label 1D shape: (650, 1250)
Label visual shape: (650, 1250, 3)
Anchor image shape: (650, 1250, 3)
Tiled 45 patches from H:\Derrame_Data\OilDataset\test\images\img_0004.jpg
Image shape: (650, 1250, 3)
Label 1D shape: (650, 1250)
Label visual shape: (650, 1250, 3)
Anchor image shape: (650, 1250, 3)
Tiled 45 patches from H:\Derrame_Data\OilDataset\test\images\img_0005.jpg


In [1]:
import os
import json
import torch
import numpy as np
from PIL import Image 
from skimage.io import imread
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import os
import cv2
import numpy as np
from skimage.io import imread
class ImagePadder:
    def __init__(self, image_shape, tile_size, stride, anchor_image):
        self.image_height, self.image_width = image_shape[:2]
        self.tile_size = tile_size
        self.stride = stride
        self.anchor_image = anchor_image

        self.pad_left, self.pad_right = self.calculate_padding(self.image_width)
        self.pad_top, self.pad_bottom = self.calculate_padding(self.image_height)

        self.padded_width = self.image_width + self.pad_left + self.pad_right
        self.padded_height = self.image_height + self.pad_top + self.pad_bottom

    def calculate_padding(self, size):
        num_tiles = ((size - self.tile_size) + self.stride - 1) // self.stride + 1
        padded_size = self.tile_size + (num_tiles - 1) * self.stride
        total_padding = padded_size - size
        pad_before = total_padding // 2
        pad_after = total_padding - pad_before
        return pad_before, pad_after

    def pad_image(self, image):
        # Resize the anchor image to the padded size
        anchor_resized = cv2.resize(
            self.anchor_image,
            (self.padded_width, self.padded_height),
            interpolation=cv2.INTER_LINEAR,
        )

        # Place the original image in the center
        anchor_resized[
            self.pad_top : self.pad_top + self.image_height,
            self.pad_left : self.pad_left + self.image_width,
        ] = image

        return anchor_resized

    def pad_label(self, label):
        if label.ndim == 2:
            # Grayscale label
            pad_width = (
                (self.pad_top, self.pad_bottom),
                (self.pad_left, self.pad_right)
            )
        elif label.ndim == 3:
            # RGB label
            pad_width = (
                (self.pad_top, self.pad_bottom),
                (self.pad_left, self.pad_right),
                (0, 0)  # No padding on the channel dimension
            )
        else:
            raise ValueError(f"Unsupported label dimensions: {label.ndim}")
        
        padded_label = np.pad(
            label,
            pad_width,
            mode='constant',
            constant_values=0  # Assuming background class is 0
        )
        return padded_label


In [10]:
from PIL import Image

# Path to your image
image_path = 'M:\Mi unidad\CIMA2023\Documentos2023\Proyectos\Proy1-ImagenesEspectrales\data\imagenes\OilDataset\\test\images\img_0001.jpg'

# Open the image
image = Image.open(image_path)

# Get image properties
print(f"Image size: {image.size}")
print(f"Image mode: {image.mode}")  # This will tell you if it's grayscale ("L") or RGB ("RGB")

# Show the image (optional)
image.show()

Image size: (1250, 650)
Image mode: RGB


STANDARIZE CLASSES LABEL_1D

In [47]:
import os

# Path to your directory
directory = 'H:/Derrame_Data/filtered_patches/labels'
# Initialize counters
sat_count = 0
mask_count = 0

# Loop through the files in the directory
for filename in os.listdir(directory):
    if filename.endswith("_sat.jpg"):
        sat_count += 1
    elif filename.endswith("label.png"):
        mask_count += 1

# Print the results
print(f"Number of satellite images: {sat_count}")
print(f"Number of mask images: {mask_count}")

Number of satellite images: 0
Number of mask images: 19096


In [22]:
import os

# Path to your directory
directory = 'H:/Derrame_Data/OilDatasetSOS/test/gt'

# Initialize counters
sat_count = 0
mask_count = 0

# Loop through the files in the directory
for filename in os.listdir(directory):
    if filename.endswith("_sat.jpg"):
        sat_count += 1
    elif filename.endswith("_mask.png"):
        mask_count += 1

# Print the results
print(f"Number of satellite images: {sat_count}")
print(f"Number of mask images: {mask_count}")

Number of satellite images: 0
Number of mask images: 839


In [27]:
from PIL import Image
import numpy as np

# Path to your image
image_path = 'H:\Derrame_Data\OilDataset\\test\labels_1D\img_0002.png'

# Open the image
image = Image.open(image_path)

# Get image properties
print(f"Image size: {image.size}")
print(f"Image mode: {image.mode}")  # This will tell you if it's grayscale ("L") or RGB ("RGB")

# Convert the image to a NumPy array
image_array = np.array(image)

# Print some pixel values
print("Pixel values:")
print(image_array)  # This will print out the pixel values in array format

# Optionally, print the shape of the image (to confirm the channels)
print(f"Image array shape: {image_array.shape}")


Image size: (1250, 650)
Image mode: L
Pixel values:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 4 4 4]
 [0 0 0 ... 4 4 4]
 [0 0 0 ... 4 4 4]]
Image array shape: (650, 1250)


In [41]:
from PIL import Image
import numpy as np

# Path to your image
image_path = 'H:\Derrame_Data\OilDatasetPatchesFiltered\\train\\labels\\img_0001_11_label.png'

# Open the image
image = Image.open(image_path)

# Get image properties
print(f"Image size: {image.size}")
print(f"Image mode: {image.mode}")  # This will tell you if it's grayscale ("L") or RGB ("RGB")

# Convert the image to a NumPy array
image_array = np.array(image)

# Print some pixel values
print("Pixel values:")
print(image_array)  # This will print out the pixel values in array format

# Get unique RGB values and their counts
# Reshape the array to have each pixel's RGB as a row
pixels = image_array.reshape(-1, image_array.shape[-1])

# Count the unique rows (RGB values) and their occurrences
unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)

# Display the results
for color, count in zip(unique_colors, counts):
    print(f"Color {color} occurs {count} times")

# Optionally, print the shape of the image (to confirm the channels)
print(f"Image array shape: {image_array.shape}")

Image size: (256, 256)
Image mode: RGB
Pixel values:
[[[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 ...

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]]
Color [0 0 0] occurs 64432 times
Color [  0 255 255] occurs 1104 times
Image array shape: (256, 256, 3)


In [30]:
def convert_sos_masks_to_labels_1d(input_dir, output_dir):
    """
    Converts RGB mask images in the SOS dataset to single-channel labels_1D format.

    Parameters:
    - input_dir: Directory containing the original mask images (e.g., 'train').
    - output_dir: Directory to save the converted labels (e.g., 'train_1D').
    """
    os.makedirs(output_dir, exist_ok=True)

    mask_files = [f for f in os.listdir(input_dir) if f.endswith('_mask.png') or f.endswith('_mask.jpg')]

    for mask_file in mask_files:
        mask_path = os.path.join(input_dir, mask_file)

        # Load the RGB mask image
        mask_rgb = Image.open(mask_path).convert('RGB')
        mask_array = np.array(mask_rgb)

        # Initialize the label array
        label_array = np.zeros((mask_array.shape[0], mask_array.shape[1]), dtype=np.uint8)

        # Create a boolean mask for white pixels (Oil Spill)
        oil_spill_mask = np.all(mask_array == [255, 255, 255], axis=-1)

        # Set Oil Spill pixels to 1
        label_array[oil_spill_mask] = 1  # Oil Spill

        # The remaining pixels are already set to 0 (Sea Surface)

        # Convert the label array to an image
        label_image = Image.fromarray(label_array, mode='L')

        # Save the label image
        label_file = mask_file.replace('_mask.png', '_label_1D.png').replace('_mask.jpg', '_label_1D.png')
        label_path = os.path.join(output_dir, label_file)
        label_image.save(label_path)

        print(f"Converted and saved: {label_path}")


In [31]:
# Paths to the SOS dataset directories
sos_train_dir = 'H:/Derrame_Data/OilDatasetSOS/train'  # Replace with your actual path
sos_train_1d_dir = 'H:/Derrame_Data/OilDatasetSOS/train_1D'  # New directory for converted masks

# Convert the masks in the training set
convert_sos_masks_to_labels_1d(sos_train_dir, sos_train_1d_dir)


Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20840_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20841_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20842_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20843_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20844_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20845_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20846_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20847_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20848_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20849_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20850_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/train_1D\20851_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/t

In [34]:
# Paths to the SOS dataset directories
sos_train_dir = 'H:/Derrame_Data/OilDatasetSOS/test/gt'  # Replace with your actual path
sos_train_1d_dir = 'H:/Derrame_Data/OilDatasetSOS/test_1D'  # New directory for converted masks

# Convert the masks in the training set
convert_sos_masks_to_labels_1d(sos_train_dir, sos_train_1d_dir)


Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20001_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20002_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20003_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20004_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20005_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20006_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20007_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20008_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20009_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20010_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20011_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20012_label_1D.png
Converted and saved: H:/Derrame_Data/OilDatasetSOS/test_1D\20013

In [32]:
import os

# Path to your directory
directory = 'H:/Derrame_Data/OilDatasetSOS/train_1D'

# Initialize counters
sat_count = 0
mask_count = 0

# Loop through the files in the directory
for filename in os.listdir(directory):
    if filename.endswith("_sat.jpg"):
        sat_count += 1
    elif filename.endswith("_label_1D.png"):
        mask_count += 1

# Print the results
print(f"Number of satellite images: {sat_count}")
print(f"Number of mask images: {mask_count}")

Number of satellite images: 0
Number of mask images: 3354


In [42]:
def filter_patches(
    input_dir,
    output_dir,
    classes_of_interest=[1, 2, 3],
    land_class=4,
    land_percentage_range=(0.25, 0.45)
):
    """
    Filters patches based on the presence of specified classes or land percentage.

    Parameters:
    - input_dir: Directory containing the patches (with subdirectories 'images', 'labels_1D', 'labels')
    - output_dir: Directory where the filtered patches will be saved
    - classes_of_interest: List of class labels to check for inclusion (default: [1, 2, 3])
    - land_class: Class label for 'Land' (default: 4)
    - land_percentage_range: Tuple indicating the inclusive range of land percentage to include (default: (0.25, 0.45))
    """
    import os
    import shutil
    from PIL import Image
    import numpy as np

    images_dir = os.path.join(input_dir, 'images')
    labels_1d_dir = os.path.join(input_dir, 'labels_1D')
    labels_dir = os.path.join(input_dir, 'labels')

    output_images_dir = os.path.join(output_dir, 'images')
    output_labels_1d_dir = os.path.join(output_dir, 'labels_1D')
    output_labels_dir = os.path.join(output_dir, 'labels')

    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_1d_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)

    label_files = [f for f in os.listdir(labels_1d_dir) if f.endswith('.png') or f.endswith('.jpg')]

    for label_file in label_files:
        label_path = os.path.join(labels_1d_dir, label_file)
        label_image = Image.open(label_path).convert('L')  # Correctly open the image
        label_array = np.array(label_image)  # Convert image to NumPy array

        # Check if label contains any of the classes of interest
        contains_interest_class = np.isin(label_array, classes_of_interest).any()

        # Compute land percentage
        total_pixels = label_array.size
        land_pixels = np.sum(label_array == land_class)
        land_percentage = land_pixels / total_pixels

        # Condition for inclusion
        if contains_interest_class or (land_percentage >= land_percentage_range[0] and land_percentage <= land_percentage_range[1]):
            # Copy the label_1D file
            shutil.copy2(label_path, os.path.join(output_labels_1d_dir, label_file))

            # Corresponding image and label files
            base_name = label_file.replace('_label_1D.png', '').replace('_label_1D.jpg', '')
            image_file_png = base_name + '.png'
            image_file_jpg = base_name + '.jpg'
            label_visual_file_png = base_name + '_label.png'
            label_visual_file_jpg = base_name + '_label.jpg'

            # Check and copy the image file
            image_path = None
            if os.path.exists(os.path.join(images_dir, image_file_png)):
                image_file = image_file_png
                image_path = os.path.join(images_dir, image_file)
            elif os.path.exists(os.path.join(images_dir, image_file_jpg)):
                image_file = image_file_jpg
                image_path = os.path.join(images_dir, image_file_jpg)
            else:
                print(f"Image file not found for label: {label_file}")
                continue  # Skip to next iteration

            shutil.copy2(image_path, os.path.join(output_images_dir, image_file))

            # Check and copy the label visual file
            label_visual_path = None
            if os.path.exists(os.path.join(labels_dir, label_visual_file_png)):
                label_visual_file = label_visual_file_png
                label_visual_path = os.path.join(labels_dir, label_visual_file)
            elif os.path.exists(os.path.join(labels_dir, label_visual_file_jpg)):
                label_visual_file = label_visual_file_jpg
                label_visual_path = os.path.join(labels_dir, label_visual_file_jpg)
            else:
                print(f"Visual label file not found for label: {label_file}")
                continue  # Skip to next iteration

            shutil.copy2(label_visual_path, os.path.join(output_labels_dir, label_visual_file))
        else:
            # Discard the patch
            pass  # Do nothing


In [43]:
# Paths to your existing patches and the desired output directory
input_dir = 'H:/Derrame_Data/OilDatasetPatches/train'  # Replace with your actual path
output_dir = 'H:/Derrame_Data/filtered_patches'  # Replace with your desired output path

filter_patches(
    input_dir=input_dir,
    output_dir=output_dir,
    classes_of_interest=[1, 2, 3],
    land_class=4,
    land_percentage_range=(0.25, 0.45)  # Include patches with 25% to 45% land
)


In [48]:
# Paths to your existing patches and the desired output directory
input_dir = 'H:/Derrame_Data/OilDatasetPatches/test'  # Replace with your actual path
output_dir = 'H:/Derrame_Data/filtered_patches/test'  # Replace with your desired output path

filter_patches(
    input_dir=input_dir,
    output_dir=output_dir,
    classes_of_interest=[1, 2, 3],
    land_class=4,
    land_percentage_range=(0.25, 0.45)  # Include patches with 25% to 45% land
)


CREATE SUBSET DATA

In [50]:
import os
import shutil
import random
random.seed(322)  # You can replace 42 with any number to set the seed
def create_subset(input_dirs, output_dir, krestininis_sample_size=120, sos_sample_size=50):
    """
    Creates a subset from the Krestininis and SOS datasets, selecting random samples from each.
    
    Parameters:
    - input_dirs: A dictionary with 'krestininis' and 'sos' as keys and their paths as values.
    - output_dir: The directory where the subset will be stored.
    - krestininis_sample_size: The number of images to select from the Krestininis dataset.
    - sos_sample_size: The number of images to select from the SOS dataset.
    """

    # Create the output directory structure
    os.makedirs(output_dir, exist_ok=True)
    output_images_dir = os.path.join(output_dir, 'images')
    output_labels_1d_dir = os.path.join(output_dir, 'labels_1D')
    output_labels_dir = os.path.join(output_dir, 'labels')

    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_1d_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)

    # Function to copy files
    def copy_files(label_file, images_dir, labels_1d_dir, labels_dir, output_images_dir, output_labels_1d_dir, output_labels_dir):
        # Copy the label_1D file
        label_path = os.path.join(labels_1d_dir, label_file)
        shutil.copy2(label_path, os.path.join(output_labels_1d_dir, label_file))

        # Corresponding image and label files
        base_name = label_file.replace('_label_1D.png', '').replace('_label_1D.jpg', '')
        image_file_png = base_name + '.png'
        image_file_jpg = base_name + '.jpg'
        label_visual_file_png = base_name + '_label.png'
        label_visual_file_jpg = base_name + '_label.jpg'

        # Check and copy the image file
        image_path = None
        if os.path.exists(os.path.join(images_dir, image_file_png)):
            image_file = image_file_png
            image_path = os.path.join(images_dir, image_file)
        elif os.path.exists(os.path.join(images_dir, image_file_jpg)):
            image_file = image_file_jpg
            image_path = os.path.join(images_dir, image_file_jpg)

        if image_path:
            shutil.copy2(image_path, os.path.join(output_images_dir, image_file))

        # Check and copy the label visual file
        label_visual_path = None
        if os.path.exists(os.path.join(labels_dir, label_visual_file_png)):
            label_visual_file = label_visual_file_png
            label_visual_path = os.path.join(labels_dir, label_visual_file)
        elif os.path.exists(os.path.join(labels_dir, label_visual_file_jpg)):
            label_visual_file = label_visual_file_jpg
            label_visual_path = os.path.join(labels_dir, label_visual_file_jpg)

        if label_visual_path:
            shutil.copy2(label_visual_path, os.path.join(output_labels_dir, label_visual_file))

    # Process Krestininis dataset
    krestininis_labels_1d_dir = os.path.join(input_dirs['krestininis'], 'labels_1D')
    krestininis_images_dir = os.path.join(input_dirs['krestininis'], 'images')
    krestininis_labels_dir = os.path.join(input_dirs['krestininis'], 'labels')

    krestininis_label_files = [f for f in os.listdir(krestininis_labels_1d_dir) if f.endswith('_label_1D.png') or f.endswith('_label_1D.jpg')]
    krestininis_selected_files = random.sample(krestininis_label_files, krestininis_sample_size)

    for label_file in krestininis_selected_files:
        copy_files(label_file, krestininis_images_dir, krestininis_labels_1d_dir, krestininis_labels_dir, output_images_dir, output_labels_1d_dir, output_labels_dir)

    # Process SOS dataset
    sos_labels_dir = os.path.join(input_dirs['sos'], 'train')  # Labels are in the 'train' directory with '_mask.png'
    sos_images_dir = os.path.join(input_dirs['sos'], 'train')  # Images are in the same 'train' directory with '_sat.jpg'
    sos_labels_1d_dir = os.path.join(input_dirs['sos'], 'train_1D')  # Labels_1D are in 'train_1D'

    sos_mask_files = [f for f in os.listdir(sos_labels_dir) if f.endswith('_mask.png')]
    sos_selected_files = random.sample(sos_mask_files, sos_sample_size)

    for mask_file in sos_selected_files:
        # Copy the mask (label) file
        mask_path = os.path.join(sos_labels_dir, mask_file)
        shutil.copy2(mask_path, os.path.join(output_labels_dir, mask_file))

        # Corresponding satellite image
        base_name = mask_file.replace('_mask.png', '')
        sat_file = base_name + '_sat.jpg'

        # Check and copy the satellite image file
        sat_path = os.path.join(sos_images_dir, sat_file)
        if os.path.exists(sat_path):
            shutil.copy2(sat_path, os.path.join(output_images_dir, sat_file))

        # Now, also copy the corresponding _label_1D files
        label_1d_file = base_name + '_label_1D.png'
        label_1d_path = os.path.join(sos_labels_1d_dir, label_1d_file)
        if os.path.exists(label_1d_path):
            shutil.copy2(label_1d_path, os.path.join(output_labels_1d_dir, label_1d_file))

# Example usage:
input_dirs = {
    'krestininis': 'H:\Derrame_Data\\filtered_patches\\train',
    'sos': 'H:\Derrame_Data\\OilDatasetSOS'
}
output_dir = 'H:\Derrame_Data\subset_TRAIN'

create_subset(input_dirs, output_dir, krestininis_sample_size=120, sos_sample_size=50)


In [52]:
import os
import json
import numpy as np
from skimage.io import imread

def write_dict_to_json(file_json, dict_data):
    """
    Writes a dictionary to a JSON file.
    """
    with open(file_json, "w", encoding="utf-8") as fh:
        json.dump(dict_data, fh, indent=4)
    return

def filter_images(file_list, dataset_type):
    """
    Filters the file list to include only image files based on dataset type.

    Parameters:
    - file_list: List of files in the directory.
    - dataset_type: 'sos' or 'krest' indicating the dataset being processed.

    Returns:
    - List of image files matching the criteria.
    """
    valid_extensions = ['.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif']
    if dataset_type == 'sos':
        # Include only files ending with '_sat.jpg'
        return [file for file in file_list
                if file.endswith('_sat.jpg') and
                os.path.splitext(file)[1].lower() in valid_extensions]
    elif dataset_type == 'krest':
        # Include all valid image files
        return [file for file in file_list
                if os.path.splitext(file)[1].lower() in valid_extensions]
    else:
        # If dataset_type is unknown, return an empty list
        return []

def compute_stats_for_datasets(sos_dirs, krest_dirs, file_json):
    """
    Computes the mean and std of images in the provided directories.

    Parameters:
    - sos_dirs: List of directories containing SOS images.
    - krest_dirs: List of directories containing Krestininis images.
    - file_json: Path to the JSON file where results will be saved.
    """
    all_means = []
    all_stds = []
    total_images = 0

    # Process SOS dataset
    for dir_images in sos_dirs:
        if not os.path.isdir(dir_images):
            print(f"Directory {dir_images} does not exist. Skipping.")
            continue

        image_files = sorted(filter_images(os.listdir(dir_images), dataset_type='sos'))
        num_images = len(image_files)
        total_images += num_images
        print(f"Processing {num_images} images in {dir_images} (SOS dataset)")

        for idx, image_file in enumerate(image_files):
            image_path = os.path.join(dir_images, image_file)
            image = imread(image_path)
            image = image / 255.0  # Normalize pixel values to [0, 1]

            if image.ndim == 3:
                mean_per_channel = np.mean(image, axis=(0, 1))
                std_per_channel = np.std(image, axis=(0, 1))
            elif image.ndim == 2:
                mean_per_channel = np.mean(image)
                std_per_channel = np.std(image)
                mean_per_channel = np.array([mean_per_channel])
                std_per_channel = np.array([std_per_channel])
            else:
                print(f"Skipping image {image_path}: unexpected number of dimensions ({image.ndim}).")
                continue

            all_means.append(mean_per_channel)
            all_stds.append(std_per_channel)

    # Process Krestininis dataset
    for dir_images in krest_dirs:
        if not os.path.isdir(dir_images):
            print(f"Directory {dir_images} does not exist. Skipping.")
            continue

        image_files = sorted(filter_images(os.listdir(dir_images), dataset_type='krest'))
        num_images = len(image_files)
        total_images += num_images
        print(f"Processing {num_images} images in {dir_images} (Krestininis dataset)")

        for idx, image_file in enumerate(image_files):
            image_path = os.path.join(dir_images, image_file)
            image = imread(image_path)
            image = image / 255.0  # Normalize pixel values to [0, 1]

            if image.ndim == 3:
                mean_per_channel = np.mean(image, axis=(0, 1))
                std_per_channel = np.std(image, axis=(0, 1))
            elif image.ndim == 2:
                mean_per_channel = np.mean(image)
                std_per_channel = np.std(image)
                mean_per_channel = np.array([mean_per_channel])
                std_per_channel = np.array([std_per_channel])
            else:
                print(f"Skipping image {image_path}: unexpected number of dimensions ({image.ndim}).")
                continue

            all_means.append(mean_per_channel)
            all_stds.append(std_per_channel)

    if total_images == 0:
        print("No images found in the provided directories.")
        return

    # Convert lists to numpy arrays
    all_means = np.array(all_means)
    all_stds = np.array(all_stds)

    # Compute overall mean and std across all images and channels
    mean_of_images = np.mean(all_means, axis=0)
    std_of_images = np.mean(all_stds, axis=0)

    # Display the results
    if mean_of_images.size == 3:
        print(f"Overall Mean per channel (R, G, B): {mean_of_images}")
        print(f"Overall Std per channel (R, G, B): {std_of_images}")
    else:
        print(f"Overall Mean: {mean_of_images[0]}")
        print(f"Overall Std: {std_of_images[0]}")

    # Prepare dictionary to save
    dict_stats = {
        "mean": mean_of_images.tolist(),
        "std": std_of_images.tolist()
    }

    write_dict_to_json(file_json, dict_stats)
    print(f"Image statistics saved in {file_json}")
    return

def main():
    # Directories for SOS dataset (images ending with '_sat.jpg')
    sos_dirs = [
        "H:\Derrame_Data\OilDatasetSOS\\train",
        "H:\Derrame_Data\OilDatasetSOS\\test",
    ]

    # Directories for Krestininis dataset
    krest_dirs = [
        "H:\Derrame_Data\\filtered_patches\\train\images",
        "H:\Derrame_Data\\filtered_patches\\test\images",
    ]

    file_json = "image_stats.json"

    compute_stats_for_datasets(sos_dirs, krest_dirs, file_json)

if __name__ == "__main__":
    main()


Processing 3354 images in H:\Derrame_Data\OilDatasetSOS\train (SOS dataset)
Processing 0 images in H:\Derrame_Data\OilDatasetSOS\test (SOS dataset)
Processing 19096 images in H:\Derrame_Data\filtered_patches\train\images (Krestininis dataset)
Processing 2028 images in H:\Derrame_Data\filtered_patches\test\images (Krestininis dataset)
Overall Mean per channel (R, G, B): [0.48539701 0.48539701 0.48539701]
Overall Std per channel (R, G, B): [0.17816202 0.17816202 0.17816202]
Image statistics saved in image_stats.json
